# 잡코리아 채용시장 분석

> 분석일: 2026-03-16
> 데이터: 잡코리아 크롤링 데이터 (2,148개 공고, 815개 회사)

## 목차
1. 데이터 로드 및 전처리
2. 지역별 채용 트렌드
3. 경력 요구사항 분석
4. 회사 규모별 채용 패턴
5. 키워드별 직무 특성 비교
6. 인사이트 요약

In [ ]:
# 필수 라이브러리 설치 (필요시)
# !pip install pandas matplotlib seaborn sqlalchemy psycopg2-binary python-dotenv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'  # macOS
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('라이브러리 로드 완료')

## 1. 데이터 로드

In [ ]:
# 환경변수 로드
load_dotenv('../.env')

# DB 연결
db_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

# 데이터 로드
df_jobs = pd.read_sql('SELECT * FROM job_postings', engine)
df_companies = pd.read_sql('SELECT * FROM companies', engine)

# 회사 정보 조인
df = df_jobs.merge(df_companies, left_on='company_id', right_on='id', suffixes=('', '_company'))

print(f'공고 수: {len(df_jobs):,}개')
print(f'회사 수: {len(df_companies):,}개')
print(f'조인 후: {len(df):,}개')

In [ ]:
# 데이터 미리보기
df.head()

In [ ]:
# 컬럼 정보
df.info()

## 2. 지역별 채용 트렌드

In [ ]:
# 지역 정규화 (시/구 단위로 통합)
def normalize_location(loc):
    if pd.isna(loc):
        return '기타'
    # '서울 강남구 외 5' -> '서울 강남구'
    loc = loc.split(' 외')[0]
    parts = loc.split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}"
    return loc

df['location_normalized'] = df['location'].apply(normalize_location)

# 광역시/도 추출
df['region'] = df['location_normalized'].apply(lambda x: x.split()[0] if pd.notna(x) else '기타')

In [ ]:
# 광역시/도별 채용 분포
region_counts = df['region'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', len(region_counts))
bars = ax.barh(region_counts.index[::-1], region_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('공고 수')
ax.set_title('광역시/도별 채용공고 분포 (TOP 10)')

# 값 표시
for bar, val in zip(bars, region_counts.values[::-1]):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 서울 내 구별 분포
seoul_df = df[df['region'] == '서울']
seoul_gu = seoul_df['location_normalized'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Oranges_r', len(seoul_gu))
bars = ax.barh(seoul_gu.index[::-1], seoul_gu.values[::-1], color=colors[::-1])
ax.set_xlabel('공고 수')
ax.set_title('서울시 구별 채용공고 분포 (TOP 10)')

for bar, val in zip(bars, seoul_gu.values[::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')

plt.tight_layout()
plt.show()

## 3. 경력 요구사항 분석

In [ ]:
# 경력 요구사항 분류
def categorize_experience(exp):
    if pd.isna(exp):
        return '미표기'
    exp = str(exp)
    if '신입' in exp and '경력' in exp:
        return '신입/경력'
    elif '신입' in exp:
        return '신입'
    elif '경력무관' in exp:
        return '경력무관'
    elif '경력' in exp:
        # 연차 추출
        import re
        match = re.search(r'(\d+)년', exp)
        if match:
            years = int(match.group(1))
            if years <= 3:
                return '경력 1-3년'
            elif years <= 5:
                return '경력 4-5년'
            else:
                return '경력 6년+'
        return '경력'
    return '기타'

df['exp_category'] = df['experience'].apply(categorize_experience)

In [ ]:
# 경력별 분포 파이차트
exp_counts = df['exp_category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 파이차트
colors = sns.color_palette('Set2', len(exp_counts))
axes[0].pie(exp_counts.values, labels=exp_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('경력 요구사항 분포')

# 막대차트
bars = axes[1].bar(exp_counts.index, exp_counts.values, color=colors)
axes[1].set_xlabel('경력 요구사항')
axes[1].set_ylabel('공고 수')
axes[1].set_title('경력 요구사항별 공고 수')
axes[1].tick_params(axis='x', rotation=45)

for bar, val in zip(bars, exp_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 10, f'{val:,}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# 신입 친화도 계산
newbie_friendly = df['exp_category'].isin(['신입', '신입/경력', '경력무관']).sum()
total = len(df)
newbie_ratio = newbie_friendly / total * 100

print(f'신입 지원 가능 공고: {newbie_friendly:,}개 ({newbie_ratio:.1f}%)')
print(f'경력직 전용 공고: {total - newbie_friendly:,}개 ({100-newbie_ratio:.1f}%)')

## 4. 회사 규모별 채용 패턴

In [ ]:
# 회사 규모 분포
size_counts = df['company_size'].value_counts()
size_counts = size_counts[size_counts.index.notna() & (size_counts.index != '-')]

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', len(size_counts))
bars = ax.barh(size_counts.index[::-1], size_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('회사 수')
ax.set_title('회사 규모별 분포')

for bar, val in zip(bars, size_counts.values[::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 회사 규모별 평균 채용공고 수
size_job_counts = df.groupby('company_size').size().reset_index(name='job_count')
company_counts = df.groupby('company_size')['company_id'].nunique().reset_index(name='company_count')
size_stats = size_job_counts.merge(company_counts, on='company_size')
size_stats['avg_jobs'] = size_stats['job_count'] / size_stats['company_count']
size_stats = size_stats[size_stats['company_size'].notna() & (size_stats['company_size'] != '-')]
size_stats = size_stats.sort_values('avg_jobs', ascending=False)

print('회사 규모별 평균 채용공고 수:')
print(size_stats.to_string(index=False))

In [ ]:
# 회사 규모 x 경력 요구사항 히트맵
heatmap_data = pd.crosstab(df['company_size'], df['exp_category'], normalize='index') * 100
heatmap_data = heatmap_data.loc[heatmap_data.index.notna() & (heatmap_data.index != '-')]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('회사 규모별 경력 요구사항 분포 (%)')
ax.set_xlabel('경력 요구사항')
ax.set_ylabel('회사 규모')
plt.tight_layout()
plt.show()

## 5. 키워드별 직무 특성 비교

In [ ]:
# 키워드별 분포 (None 제외)
df_with_keyword = df[df['search_keyword'].notna()]
keyword_counts = df_with_keyword['search_keyword'].value_counts()

print('키워드별 공고 수:')
for kw, cnt in keyword_counts.items():
    print(f'  {kw}: {cnt:,}개')

In [ ]:
# 키워드별 경력 요구사항 비교
keyword_exp = pd.crosstab(df_with_keyword['search_keyword'], df_with_keyword['exp_category'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 6))
keyword_exp.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('검색 키워드')
ax.set_ylabel('비율 (%)')
ax.set_title('키워드별 경력 요구사항 분포')
ax.legend(title='경력', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 키워드별 지역 TOP 5 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

keywords = keyword_counts.index[:3].tolist()
for ax, kw in zip(axes, keywords):
    kw_df = df_with_keyword[df_with_keyword['search_keyword'] == kw]
    loc_counts = kw_df['location_normalized'].value_counts().head(5)
    ax.barh(loc_counts.index[::-1], loc_counts.values[::-1], color=sns.color_palette('Greens_r', 5)[::-1])
    ax.set_title(f'{kw}\n상위 지역')
    ax.set_xlabel('공고 수')

plt.tight_layout()
plt.show()

In [ ]:
# 키워드별 회사 규모 분포
keyword_size = pd.crosstab(df_with_keyword['search_keyword'], df_with_keyword['company_size'], normalize='index') * 100
# 주요 규모만 선택
main_sizes = ['대기업', '중견기업', '중소기업', '스타트업']
keyword_size_main = keyword_size[[s for s in main_sizes if s in keyword_size.columns]]

fig, ax = plt.subplots(figsize=(12, 6))
keyword_size_main.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('검색 키워드')
ax.set_ylabel('비율 (%)')
ax.set_title('키워드별 회사 규모 분포')
ax.legend(title='회사 규모', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 6. 인사이트 요약

In [ ]:
print('=' * 60)
print('잡코리아 채용시장 분석 인사이트 요약')
print('=' * 60)
print()
print(f'1. 데이터 규모: {len(df):,}개 공고, {len(df_companies):,}개 회사')
print()
print(f'2. 지역 분포:')
print(f'   - 서울이 전체의 {(df["region"]=="서울").sum()/len(df)*100:.1f}% 차지')
print(f'   - 서울 내 강남구가 가장 많음 ({seoul_gu.iloc[0]:,}개)')
print()
print(f'3. 경력 요구사항:')
print(f'   - 신입 지원 가능: {newbie_ratio:.1f}%')
print(f'   - 가장 많은 유형: {exp_counts.index[0]} ({exp_counts.iloc[0]:,}개)')
print()
print(f'4. 회사 규모:')
print(f'   - 중소기업이 가장 많음 ({size_counts.iloc[0]:,}개)')
print(f'   - 대기업 비중: {size_counts.get("대기업", 0)/size_counts.sum()*100:.1f}%')
print()
print('=' * 60)